In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [34]:
clean_file_path = 'codebar-AI-fundamentals-pixels-banking-ML-model/data/pixels_banking_loan_repayment_clean.csv'
df_model_ready = pd.read_csv(clean_file_path)

In [35]:
X = df_model_ready.drop(columns=['loan_repaid'])
y = df_model_ready['loan_repaid']

# Split into Training and Testing sets (20% test size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training features shape:', X_train.shape)
print('Testing features shape:', X_test.shape)

Training features shape: (960, 39)
Testing features shape: (240, 39)


In [36]:
# List the columns that need to be scaled down
numeric_cols_to_scale = [
    'age', 'tenure_months', 'monthly_income', 'avg_monthly_balance', 
    'credit_score', 'num_products', 'customer_service_calls', 
    'missed_payments_6m', 'support_rating', 'monthly_fees', 
    'digital_engagement_score', 'loan_amount', 'loan_term_months', 
    'debt_to_income_ratio', 'interest_rate'
]

# Initialize the scaler
scaler = StandardScaler()

# Create copies so we don't disrupt our original split dataframes
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Secure Rule: fit_transform training data, only transform test data
X_train_scaled[numeric_cols_to_scale] = scaler.fit_transform(X_train[numeric_cols_to_scale])
X_test_scaled[numeric_cols_to_scale] = scaler.transform(X_test[numeric_cols_to_scale])

print("Features successfully scaled!")

Features successfully scaled!


In [37]:
model = LogisticRegression(max_iter=1000, solver='liblinear')

In [38]:
model.fit(X_train_scaled, y_train)
print('Model trained successfully on scaled data')

Model trained successfully on scaled data


In [39]:
y_pred = model.predict(X_test_scaled)

y_pred[:10]

array([0, 1, 1, 0, 0, 1, 0, 1, 1, 1])

In [40]:
results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

results.head(10)

,Actual,Predicted
0,0,0
1,1,1
2,0,1
3,0,0
4,0,0
5,1,1
6,0,0
7,1,1
8,1,1
9,0,1


In [41]:
accuracy = accuracy_score(y_test, y_pred)

print('Accuracy:', round(accuracy, 3))
print('Accuracy percentage:', round(accuracy * 100, 1), '%')

Accuracy: 0.754
Accuracy percentage: 75.4 %


In [42]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=['Actual Not Repaid (0)', 'Actual Repaid (1)'],
    columns=['Predicted Not Repaid (0)', 'Predicted Repaid (1)']
)

cm_df

,Predicted Not Repaid (0),Predicted Repaid (1)
Actual Not Repaid (0),31,43
Actual Repaid (1),16,150


In [43]:
print("\n--- Corrected Confusion Matrix ---")
print(cm_df)


--- Corrected Confusion Matrix ---
                       Predicted Not Repaid (0)  Predicted Repaid (1)
Actual Not Repaid (0)                        31                    43
Actual Repaid (1)                            16                   150


In [44]:
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))


--- Detailed Classification Report ---
              precision    recall  f1-score   support

           0       0.66      0.42      0.51        74
           1       0.78      0.90      0.84       166

    accuracy                           0.75       240
   macro avg       0.72      0.66      0.67       240
weighted avg       0.74      0.75      0.74       240



In [45]:
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print(coefficients)

                            Feature  Coefficient
4                      credit_score     0.718539
28            account_type_Everyday     0.670684
35          uses_mobile_app_Unknown     0.585293
19                region_North West     0.423896
29                account_type_Plus     0.406036
36              uses_mobile_app_Yes     0.402671
16       monthly_income_was_missing     0.341864
15         credit_score_was_missing     0.326067
24                   region_Unknown     0.320467
2                    monthly_income     0.299507
17                    region_London     0.282532
9                      monthly_fees     0.235434
1                     tenure_months     0.194793
18                region_North East     0.181842
31             account_type_Unknown     0.109762
5                      num_products     0.069479
23                region_South West     0.067544
14                    interest_rate     0.067133
33              has_credit_card_Yes     0.044729
10         digital_e